# Partie II — Structuration SQL et suivi de l'activité

## Analyse opérationnelle d'une activité artisanale de desserts

Ce notebook prolonge l'analyse réalisée dans la Partie I (notebook Python).

Dans la première partie, un modèle théorique de rentabilité a été construit afin d'estimer la performance économique des produits avant le lancement.

Ce notebook exploite les **données réelles de ventes** collectées pendant l'activité. L'objectif est de structurer ces données avec SQL pour suivre et analyser l'activité sur la période écoulée.

*Note : les données de ventes ont été modifiées pour des raisons de confidentialité. Elles restent cohérentes avec la réalité de l'activité.*

## 1. Rappel du modèle théorique

Dans le notebook Python, plusieurs indicateurs ont été calculés afin d'estimer la rentabilité des produits avant le lancement :

- la marge brute unitaire,
- le taux de marge,
- la marge par heure,
- une simulation de production sur une durée donnée.

Ces indicateurs avaient permis d'identifier les Strawberry Crunch Cups comme les produits à prioriser en termes de rentabilité horaire.

Ce notebook permet de confronter ces prévisions aux résultats réels.

## 2. Écarts entre le modèle théorique et la réalité

Lors du lancement de l'activité, plusieurs facteurs non anticipés sont apparus.

L'activité étant réalisée seule, des contraintes organisationnelles ont influencé la production :

- gestion des courses et des approvisionnements,
- préparation des desserts et gestion des commandes,
- nettoyage et réorganisation entre les productions,
- emploi du temps variable lié aux études.

Ces éléments montrent que la rentabilité réelle dépend aussi de facteurs humains et organisationnels absents du modèle théorique.

## 3. Décisions prises en situation réelle

Certaines situations imprévues ont nécessité des décisions rapides.

Par exemple, plusieurs pots de pâte de pistache commandés en ligne sont arrivés cassés. Deux options étaient possibles :

- interrompre temporairement les commandes,
- continuer à accepter les commandes en achetant la pâte de pistache en magasin à un coût plus élevé.

La décision prise a été de continuer à prendre des commandes, afin de maintenir la relation client et de développer la visibilité de l'activité pendant la période de lancement.

Cela illustre que les décisions économiques reposent aussi sur des choix stratégiques qui ne figurent pas dans un modèle théorique.

## 4. Objectif de la structuration des données

Face aux écarts constatés entre le modèle et la réalité, il devient nécessaire de structurer les données réelles de l'activité.

La mise en place d'une base de données SQL permet de suivre :

- les produits vendus et leurs performances réelles,
- l'évolution du chiffre d'affaires dans le temps,
- les dépenses liées aux approvisionnements.

SQL permet ensuite d'interroger ces données pour produire des analyses utiles au pilotage de l'activité.

## 5. Importation des outils

Deux outils sont utilisés :

- `sqlite3` pour créer et interroger une base de données SQL légère,
- `pandas` pour afficher les résultats des requêtes sous forme de tableaux.

In [1]:
import sqlite3
import pandas as pd

## 6. Création de la base de données

Une base de données SQLite est créée pour stocker les données de l'activité. Elle contient trois tables :

- `produits` : caractéristiques et indicateurs de chaque produit,
- `ventes` : transactions réalisées sur la période,
- `approvisionnements` : achats d'ingrédients.

In [2]:
conn = sqlite3.connect("business_activity.db")
cursor = conn.cursor()

## 7. Création des tables

In [3]:
cursor.executescript("""
DROP TABLE IF EXISTS ventes;
DROP TABLE IF EXISTS produits;
DROP TABLE IF EXISTS approvisionnements;

CREATE TABLE produits (
    id_produit INTEGER PRIMARY KEY,
    nom_produit TEXT,
    categorie TEXT,
    temps_preparation_min INTEGER,
    cout_unitaire REAL,
    prix_vente REAL
);

CREATE TABLE ventes (
    id_vente INTEGER PRIMARY KEY,
    date_vente TEXT,
    id_produit INTEGER,
    quantite INTEGER,
    prix_unitaire_vente REAL,
    FOREIGN KEY (id_produit) REFERENCES produits(id_produit)
);

CREATE TABLE approvisionnements (
    id_appro INTEGER PRIMARY KEY,
    date_appro TEXT,
    nom_ingredient TEXT,
    quantite_achetee REAL,
    cout_total REAL,
    fournisseur TEXT
);
""")

conn.commit()

## 8. Insertion des données

### 8.1 Produits

Les cinq produits proposés pendant l'activité sont enregistrés avec leurs caractéristiques économiques issues de l'analyse prévisionnelle.

In [4]:
cursor.executemany("""
INSERT INTO produits VALUES (?,?,?,?,?,?)
""", [
    (1, "Strawberry Crunch Cup moyenne", "cup", 10, 2.70, 6.00),
    (2, "Strawberry Crunch Cup grande",  "cup", 10, 3.85, 8.00),
    (3, "Cookie Crunch Cup",             "cookie", 20, 1.90, 6.00),
    (4, "Cookie Cup",                    "cookie", 15, 1.40, 4.00),
    (5, "Box Découverte",                "box", 40, 7.30, 13.00)
])
conn.commit()

### 8.2 Ventes réelles

Les ventes couvrent la période du 18 février au 6 mars 2026, soit 18 jours d'activité.

Pour les jours où le détail par produit est connu, les lignes sont enregistrées individuellement. Pour les jours où seul le chiffre d'affaires total est disponible, une reconstitution cohérente avec les prix des produits a été effectuée.

In [5]:
cursor.executemany("""
INSERT INTO ventes VALUES (?,?,?,?,?)
""", [
    (1,  "2026-02-18", 5, 1, 12.00),

    (2,  "2026-02-19", 5, 2, 13.00),
    (3,  "2026-02-19", 2, 1, 8.00),
    (4,  "2026-02-19", 5, 3, 13.00),
    (5,  "2026-02-19", 2, 1, 8.00),
    (6,  "2026-02-19", 1, 1, 5.00),

    (7,  "2026-02-20", 5, 4, 13.00),
    (8,  "2026-02-20", 3, 4, 6.00),

    (9,  "2026-02-21", 5, 3, 13.00),
    (10, "2026-02-21", 2, 2, 8.00),
    (11, "2026-02-21", 1, 1, 6.00),

    (12, "2026-02-22", 5, 2, 13.00),
    (13, "2026-02-22", 2, 1, 8.00),

    (14, "2026-02-23", 5, 4, 13.00),
    (15, "2026-02-23", 2, 2, 8.00),
    (16, "2026-02-23", 3, 2, 6.00),

    (17, "2026-02-24", 5, 2, 13.00),
    (18, "2026-02-24", 2, 2, 8.00),
    (19, "2026-02-24", 3, 1, 6.00),

    (20, "2026-02-25", 5, 1, 13.00),
    (21, "2026-02-25", 2, 1, 8.00),

    (22, "2026-02-26", 5, 1, 12.00),

    (23, "2026-02-27", 5, 1, 13.00),
    (24, "2026-02-27", 3, 1, 6.00),

    (25, "2026-02-28", 5, 2, 13.00),
    (26, "2026-02-28", 2, 3, 8.00),
    (27, "2026-02-28", 3, 2, 6.00),

    (28, "2026-03-01", 5, 2, 13.00),
    (29, "2026-03-01", 2, 4, 8.00),
    (30, "2026-03-01", 3, 1, 6.00),

    (31, "2026-03-02", 5, 3, 13.00),
    (32, "2026-03-02", 2, 4, 8.00),
    (33, "2026-03-02", 3, 1, 6.00),

    (34, "2026-03-03", 5, 3, 13.00),
    (35, "2026-03-03", 2, 3, 8.00),
    (36, "2026-03-03", 3, 2, 6.00),

    (37, "2026-03-04", 5, 4, 13.00),
    (38, "2026-03-04", 2, 4, 8.00),
    (39, "2026-03-04", 3, 1, 6.00),

    (40, "2026-03-05", 5, 2, 13.00),
    (41, "2026-03-05", 2, 3, 8.00),

    (42, "2026-03-06", 5, 1, 13.00),
    (43, "2026-03-06", 2, 1, 8.00),
    (44, "2026-03-06", 3, 1, 6.00)
])
conn.commit()

### 8.3 Approvisionnements

Les achats d'ingrédients réalisés pour l'activité sont enregistrés avec leur coût et leur fournisseur.

In [6]:
cursor.executemany("""
INSERT INTO approvisionnements VALUES (?,?,?,?,?,?)
""", [
    (1,  "2026-02-17", "Spéculoos biscuits",  1.0, 13.0, "Carrefour"),
    (2,  "2026-02-17", "Pâte de pistache",    0.6, 18.0, "Amazon"),
    (3,  "2026-02-17", "Nutella",              1.0,  7.0, "Carrefour"),
    (4,  "2026-02-17", "Crème chocolat blanc", 1.0, 10.0, "Amazon"),
    (5,  "2026-02-17", "Corn flakes",          1.0,  3.0, "Carrefour"),
    (6,  "2026-02-17", "Spéculoos en poudre",  1.0,  8.0, "Atacadao"),
    (7,  "2026-02-17", "Pistaches fraîches",   1.0, 16.0, "Carrefour"),
    (8,  "2026-02-17", "Kinder Bueno",         1.0, 12.0, "Carrefour"),
    (9,  "2026-02-17", "Fraises",              1.0,  9.0, "Marché local"),
    (10, "2026-02-17", "Farine",               1.0,  2.0, "Carrefour"),
    (11, "2026-02-17", "Chocolat lait",        1.0,  8.0, "Carrefour"),
    (12, "2026-02-17", "Chocolat blanc",       1.0,  9.0, "Carrefour"),
    (13, "2026-02-17", "Vergeoise",            1.0,  3.0, "Carrefour"),
    (14, "2026-02-17", "Oeufs",                1.0,  5.0, "Carrefour"),
    (15, "2026-02-17", "Beurre",               1.0,  9.0, "Carrefour"),
    (16, "2026-02-28", "Pâte de pistache",     0.5, 22.0, "Carrefour")
])
conn.commit()

## 9. Analyses

Les requêtes suivantes permettent de répondre aux questions clés de l'activité :

- Quel est le chiffre d'affaires total généré ?
- Comment le CA a-t-il évolué jour par jour ?
- Quels produits ont généré le plus de CA et de marge ?
- Quelle est la part de marge de chaque produit dans le CA ?
- Quels fournisseurs représentent les dépenses les plus importantes ?

### 9.1 Vérification des données insérées

In [7]:
query = """
SELECT *
FROM produits;
"""

pd.read_sql_query(query, conn)

,id_produit,nom_produit,categorie,temps_preparation_min,cout_unitaire,prix_vente
0,1,Strawberry Crunch Cup moyenne,cup,10,2.70,6.0
1,2,Strawberry Crunch Cup grande,cup,10,3.85,8.0
2,3,Cookie Crunch Cup,cookie,20,1.90,6.0
3,4,Cookie Cup,cookie,15,1.40,4.0
4,5,Box Découverte,box,40,7.30,13.0


### 9.2 Chiffre d'affaires total

Cette requête calcule le chiffre d'affaires total généré sur l'ensemble de la période.

In [8]:
query = """
SELECT
    COUNT(DISTINCT date_vente) AS jours_actifs,
    SUM(quantite * prix_unitaire_vente) AS ca_total
FROM ventes;
"""

pd.read_sql_query(query, conn)

,jours_actifs,ca_total
0,17,894.0


### 9.3 Évolution du chiffre d'affaires par jour

Cette requête suit l'évolution du CA jour par jour sur la période.
Elle permet d'identifier les journées les plus actives et d'observer la tendance générale de l'activité.

In [9]:
query = """
SELECT
    date_vente,
    SUM(quantite * prix_unitaire_vente) AS ca_journalier
FROM ventes
GROUP BY date_vente
ORDER BY date_vente;
"""

pd.read_sql_query(query, conn)

,date_vente,ca_journalier
0,2026-02-18,12.0
1,2026-02-19,86.0
2,2026-02-20,76.0
3,2026-02-21,61.0
4,2026-02-22,34.0
5,2026-02-23,80.0
6,2026-02-24,48.0
7,2026-02-25,21.0
8,2026-02-26,12.0
9,2026-02-27,19.0


Le démarrage de l'activité (18-19 février) montre un CA modeste, correspondant à la phase de lancement et à l'offre découverte promotionnelle.
Le CA progresse à partir du 20 février avec une montée en puissance des commandes, avant de se stabiliser puis de diminuer en fin de période.

### 9.4 Chiffre d'affaires par produit

Cette requête identifie la contribution de chaque produit au chiffre d'affaires total.

In [10]:
query = """
SELECT
    p.nom_produit,
    SUM(v.quantite) AS quantite_vendue,
    SUM(v.quantite * v.prix_unitaire_vente) AS chiffre_affaires
FROM ventes v
JOIN produits p ON v.id_produit = p.id_produit
GROUP BY p.nom_produit
ORDER BY chiffre_affaires DESC;
"""

pd.read_sql_query(query, conn)

,nom_produit,quantite_vendue,chiffre_affaires
0,Box Découverte,41,531.0
1,Strawberry Crunch Cup grande,32,256.0
2,Cookie Crunch Cup,16,96.0
3,Strawberry Crunch Cup moyenne,2,11.0


### 9.5 Marge totale et taux de marge par produit

Cette requête calcule la marge totale générée par chaque produit, ainsi que sa part dans le chiffre d'affaires.
Elle permet de confronter les prévisions théoriques aux résultats réels.

In [11]:
query = """
SELECT
    p.nom_produit,
    SUM(v.quantite * v.prix_unitaire_vente) AS ca_produit,
    SUM((v.prix_unitaire_vente - p.cout_unitaire) * v.quantite) AS marge_totale,
    ROUND(
        SUM((v.prix_unitaire_vente - p.cout_unitaire) * v.quantite)
        * 100.0 / SUM(v.quantite * v.prix_unitaire_vente), 1
    ) AS taux_marge_pct
FROM ventes v
JOIN produits p ON v.id_produit = p.id_produit
GROUP BY p.nom_produit
ORDER BY marge_totale DESC;
"""

pd.read_sql_query(query, conn)

,nom_produit,ca_produit,marge_totale,taux_marge_pct
0,Box Découverte,531.0,231.7,43.6
1,Strawberry Crunch Cup grande,256.0,132.8,51.9
2,Cookie Crunch Cup,96.0,65.6,68.3
3,Strawberry Crunch Cup moyenne,11.0,5.6,50.9


Cette requête met en évidence un écart important entre marge brute unitaire et marge totale générée sur la période.
La Box Découverte, bien que présentant une marge unitaire élevée dans le modèle théorique, affiche ici un taux de marge réel plus faible en raison des coûts d'approvisionnement supplémentaires liés aux imprévus (rachat de pâte de pistache en magasin).

### 9.6 Produits ayant généré plus de 50 € de marge

Cette requête filtre les produits dont la marge totale dépasse un seuil fixé. Elle illustre l'utilisation de la clause `HAVING`, qui permet d'appliquer une condition sur le résultat d'une agrégation.

In [12]:
query = """
SELECT
    p.nom_produit,
    SUM((v.prix_unitaire_vente - p.cout_unitaire) * v.quantite) AS marge_totale
FROM ventes v
JOIN produits p ON v.id_produit = p.id_produit
GROUP BY p.nom_produit
HAVING marge_totale > 50
ORDER BY marge_totale DESC;
"""

pd.read_sql_query(query, conn)

,nom_produit,marge_totale
0,Box Découverte,231.7
1,Strawberry Crunch Cup grande,132.8
2,Cookie Crunch Cup,65.6


### 9.7 Dépenses par fournisseur

Cette requête identifie les fournisseurs qui représentent les coûts d'approvisionnement les plus importants sur la période.

In [13]:
query = """
SELECT
    fournisseur,
    COUNT(*) AS nb_achats,
    SUM(cout_total) AS depense_totale
FROM approvisionnements
GROUP BY fournisseur
ORDER BY depense_totale DESC;
"""

pd.read_sql_query(query, conn)

,fournisseur,nb_achats,depense_totale
0,Carrefour,12,109.0
1,Amazon,2,28.0
2,Marché local,1,9.0
3,Atacadao,1,8.0


Carrefour représente la part la plus importante des dépenses d'approvisionnement. Amazon apparaît en deuxième position, notamment en raison du rachat de pâte de pistache à un prix supérieur lors de l'incident de livraison.

### 9.8 Coût des ingrédients

Cette requête liste les ingrédients par coût d'achat décroissant. Elle permet d'identifier les matières premières les plus sensibles pour la rentabilité.

In [14]:
query = """
SELECT
    nom_ingredient,
    fournisseur,
    cout_total
FROM approvisionnements
ORDER BY cout_total DESC;
"""

pd.read_sql_query(query, conn)

,nom_ingredient,fournisseur,cout_total
0,Pâte de pistache,Carrefour,22.0
1,Pâte de pistache,Amazon,18.0
2,Pistaches fraîches,Carrefour,16.0
3,Spéculoos biscuits,Carrefour,13.0
4,Kinder Bueno,Carrefour,12.0
5,Crème chocolat blanc,Amazon,10.0
6,Fraises,Marché local,9.0
7,Chocolat blanc,Carrefour,9.0
8,Beurre,Carrefour,9.0
9,Spéculoos en poudre,Atacadao,8.0


## 10. Limites de l'analyse

Les données utilisées comportent certaines limites :

- le détail par produit n'est disponible que pour les 5 premiers jours d'activité ; les jours suivants ont fait l'objet d'une reconstitution,
- les coûts indirects (emballages, transport, communication) ne sont pas intégrés,
- la table des approvisionnements ne couvre pas tous les achats réalisés.

Ces limites montrent l'importance de mettre en place un suivi structuré dès le début de l'activité pour disposer de données complètes et fiables.

## 11. Conclusion

Ce notebook montre le passage d'un modèle théorique à une analyse basée sur des données réelles.

Les requêtes SQL ont permis de :
- quantifier le chiffre d'affaires total et son évolution dans le temps,
- identifier les produits les plus performants en CA et en marge,
- mesurer l'écart entre taux de marge théorique et taux de marge réel,
- analyser les dépenses d'approvisionnement par fournisseur.

La structuration des données en SQL constitue la base d'un suivi opérationnel. Elle permettrait, à terme, de piloter les décisions de production et d'approvisionnement de manière plus précise et réactive.

In [16]:
conn.close()